# Duffing Integrator Audit

Pre-registered diagnostic comparing this project's existing manual
forward-Euler Duffing generator against a `solve_ivp`/RK45 reference,
using the same style of check that caught the Harmonic oscillator
integrator bug (Section 18 of the experiment log).

**Motivation:** `simulate_duffing` uses the identical manual-Euler
coding pattern (`x_new = x + v*dt; v_new = v + ax*dt`) that produced
the Harmonic amplitude-growth bug, and had never been independently
audited. This notebook is that audit.

**Pre-registered checks and thresholds (fixed before running):**
1. **Within-512-window amplitude growth** — std ratio, second half vs.
   first half of one context window (Harmonic's own disqualifying value
   was 1.37x). Threshold: ≥1.3x flagged.
2. **Long-trajectory marginal statistics** — relative deviation in std
   between the Euler trajectory and the RK45 reference, transient
   discarded from both. Threshold: >15% flagged.
3. **Power spectrum** (descriptive only) — qualitative check for
   spurious high-frequency content.

**Decision rule:** if check 1 alone fires, that alone warrants a full
stable-generator re-evaluation, matching the Harmonic precedent.


## Setup — both generators, copied verbatim from `new_experiments.ipynb` / Exp 19

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.stats import kurtosis

SEED = 42
N_STEPS = 4000
TRANSIENT = 500  # matches the log's existing discard convention
CONTEXT_LEN = 512


In [ ]:
def simulate_duffing_euler(n_steps=4000, delta=0.3, alpha=-1.0,
                            beta=1.0, gamma=0.37, omega=1.2, seed=SEED):
    """Duffing oscillator: nonlinear, weakly chaotic. Manual forward Euler.
    Verbatim from new_experiments.ipynb / Exp 19 -- not reconstructed."""
    rng = np.random.default_rng(seed)
    dt  = 2*np.pi / omega / 50
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    t    = 0.0
    for _ in range(n_steps):
        traj.append(x)
        ax    = -delta*v - alpha*x - beta*x**3 + gamma*np.cos(omega*t)
        x_new = x + v*dt
        v_new = v + ax*dt
        x, v, t = x_new, v_new, t+dt
    return np.array(traj, dtype=np.float32), dt


In [ ]:
def simulate_duffing_reference(n_steps=4000, delta=0.3, alpha=-1.0,
                                beta=1.0, gamma=0.37, omega=1.2, seed=SEED):
    """Duffing oscillator via solve_ivp/RK45, matched IC and time span,
    tight tolerance (rtol=atol=1e-9)."""
    rng = np.random.default_rng(seed)
    dt  = 2*np.pi / omega / 50
    x0, v0 = float(rng.standard_normal()), float(rng.standard_normal())
    t_span_end = n_steps * dt
    t_eval = np.arange(n_steps) * dt

    def rhs(t, state):
        x, v = state
        ax = -delta*v - alpha*x - beta*x**3 + gamma*np.cos(omega*t)
        return [v, ax]

    sol = solve_ivp(rhs, [0, t_span_end], [x0, v0], t_eval=t_eval,
                     method='RK45', rtol=1e-9, atol=1e-9)
    if not sol.success:
        raise RuntimeError(f"Reference solve_ivp failed: {sol.message}")
    return sol.y[0].astype(np.float32), dt


## Run both generators, matched IC (same seed)

In [ ]:
euler_traj, dt_euler = simulate_duffing_euler(n_steps=N_STEPS, seed=SEED)
ref_traj, dt_ref = simulate_duffing_reference(n_steps=N_STEPS, seed=SEED)

euler_post = euler_traj[TRANSIENT:]
ref_post   = ref_traj[TRANSIENT:]

print(f"Euler trajectory: {euler_traj.shape}, dt={dt_euler:.5f}")
print(f"RK45 reference:   {ref_traj.shape}, dt={dt_ref:.5f}")


## Check 1 — within-512-window amplitude growth (Harmonic's exact diagnostic)

In [ ]:
euler_window = euler_post[:CONTEXT_LEN]
ref_window   = ref_post[:CONTEXT_LEN]

def window_growth_ratio(window):
    half = len(window) // 2
    std_first  = window[:half].std()
    std_second = window[half:].std()
    return std_second / (std_first + 1e-12)

euler_growth = window_growth_ratio(euler_window)
ref_growth   = window_growth_ratio(ref_window)

print(f"Euler generator  : first-half std={euler_window[:256].std():.4f}  "
      f"second-half std={euler_window[256:].std():.4f}  ratio={euler_growth:.4f}")
print(f"RK45 reference   : first-half std={ref_window[:256].std():.4f}  "
      f"second-half std={ref_window[256:].std():.4f}  ratio={ref_growth:.4f}")

CHECK1_THRESHOLD = 1.3
check1_flag = euler_growth >= CHECK1_THRESHOLD
print(f"\nThreshold: {CHECK1_THRESHOLD}x  |  Euler ratio: {euler_growth:.4f}  "
      f"|  FLAGGED: {check1_flag}")


## Check 2 — long-trajectory marginal statistics

In [ ]:
euler_std = euler_post.std()
ref_std   = ref_post.std()
rel_dev_std = abs(euler_std - ref_std) / ref_std

euler_kurt = kurtosis(euler_post)
ref_kurt   = kurtosis(ref_post)

print(f"Euler std={euler_std:.4f}  RK45 std={ref_std:.4f}  "
      f"relative deviation={rel_dev_std*100:.2f}%")
print(f"Euler kurtosis={euler_kurt:.4f}  RK45 kurtosis={ref_kurt:.4f}")

CHECK2_THRESHOLD = 0.15
check2_flag = rel_dev_std > CHECK2_THRESHOLD
print(f"\nThreshold: {CHECK2_THRESHOLD*100:.0f}%  |  Observed: {rel_dev_std*100:.2f}%  "
      f"|  FLAGGED: {check2_flag}")


## Check 3 — power spectrum comparison (descriptive only)

In [ ]:
def spectral_summary(traj, dt):
    freqs = np.fft.rfftfreq(len(traj), d=dt)
    power = np.abs(np.fft.rfft(traj - traj.mean()))**2
    power_norm = power / power.sum()
    omega_drive = 1.2 / (2*np.pi)  # convert to Hz-equivalent
    hf_mask = freqs > 5 * omega_drive
    return power_norm[hf_mask].sum()

euler_hf = spectral_summary(euler_post, dt_euler)
ref_hf   = spectral_summary(ref_post, dt_ref)

print(f"High-frequency power fraction (>5x drive freq):")
print(f"  Euler generator: {euler_hf*100:.4f}%")
print(f"  RK45 reference : {ref_hf*100:.4f}%")
print(f"  Ratio (euler/ref): {euler_hf/(ref_hf+1e-12):.2f}x")
print("(Descriptive only -- both values near-zero, ratio is not a reliable")
print(" standalone signal; supports checks 1/2 rather than triggering alone)")


## Verdict

In [ ]:
any_flag = check1_flag or check2_flag
print(f"Check 1 (within-window growth) flagged: {check1_flag}")
print(f"Check 2 (long-run statistics)  flagged: {check2_flag}")
print()
if any_flag:
    print(">>> AT LEAST ONE CHECK FLAGGED. Per pre-registered decision rule,")
    print(">>> this warrants a full stable-generator re-evaluation of Exp 19/")
    print(">>> Section 20 Duffing result, analogous to Section 18's Harmonic fix.")
else:
    print(">>> NO CHECKS FLAGGED. Duffing's existing Euler generator appears")
    print(">>> numerically adequate at this dt. Current result stands without")
    print(">>> a generator-stability caveat.")
